# GPU risk tests on a Kaggle T4

Four yes/no tests on the exact hardware Nova will use. Each test records PASS or FAIL and keeps going,
so one **Save & Run All** gives a full report. Expect about 45–75 minutes.

| # | Test | Pass when | If it fails |
|---|---|---|---|
| 1 | Generate in Transformers on a T4 | Coherent reply that ends with `<\|im_end\|>` | Fix versions/dtype (max 60 min), else switch to text-only Qwen3-4B or Qwen3-1.7B |
| 2 | 50-step Unsloth LoRA on 200 Bitext rows, save, reload, generate | Loss falls; the reloaded adapter changes the output | Switch to text-only Qwen3-4B or Qwen3-1.7B |
| 3 | vLLM serves the base model and returns one parsed tool call | `message.tool_calls` is filled | Evals with Transformers or llama.cpp; serve on an L4/A10-class GPU |
| 4 | vLLM serves the test adapter as a LoRA module | `model="nova-test"` answers in the adapter's style | Merge the adapter into the base model |

**Before running (Kaggle):** Settings → Accelerator **GPU T4 x2**, Internet **On**; Add-ons → Secrets → tick `HF_TOKEN`.
Then **Save Version → Save & Run All**. When it finishes, open the version's **Logs** (or Output) and copy everything
from `===== RISK TEST REPORT =====` to the end.

Tests 1–2 run on GPU 0 in this notebook. vLLM (tests 3–4) runs in its own Python environment, on GPU 1 when there are
two GPUs, so its libraries cannot break the training libraries and the two sets of versions are pinned separately.

In [ ]:
# 0. Setup: report helper, GPU plan, Hugging Face login
import gc, json, os, subprocess, sys, time, traceback

N_GPUS = len(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip().splitlines())
os.environ["CUDA_VISIBLE_DEVICES"] = "0"            # this notebook (tests 1-2) uses GPU 0
VLLM_GPU = "1" if N_GPUS >= 2 else "0"               # vLLM gets the second T4 when there is one
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout, "vLLM will use GPU", VLLM_GPU)

REPORT = []
def record(test, ok, detail, fallback=""):
    REPORT.append({"test": test, "result": "PASS" if ok else "FAIL",
                   "detail": str(detail)[-2000:], "fallback": "" if ok else fallback})
    print(f"\n### {test}: {'PASS' if ok else 'FAIL'}\n{detail}")

def err():
    return traceback.format_exc()[-1500:]

try:                                   # Kaggle
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except ImportError:                    # Colab
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN
from huggingface_hub import login
login(token=HF_TOKEN)

SYSTEM = "You are Nova, NimbusCart's support assistant."
TOOLS = [{"type": "function", "function": {
    "name": "get_order_status",
    "description": "Status of one of the signed-in customer's orders",
    "parameters": {"type": "object", "properties": {"order_id": {"type": "integer"}}, "required": ["order_id"]}}}]
ADAPTER_DIR = "/kaggle/working/spike-adapter" if os.path.isdir("/kaggle/working") else os.path.abspath("spike-adapter")

In [ ]:
# 1. Install the training stack (Unsloth brings matching transformers / trl / peft)
%pip install -q -U unsloth
%pip install -q openai

import importlib.metadata as md
def versions(names):
    found = {}
    for n in names:
        try:
            found[n] = md.version(n)
        except md.PackageNotFoundError:
            pass
    return found

TRAIN_VERSIONS = versions(["torch", "transformers", "trl", "peft", "unsloth", "unsloth_zoo",
                           "accelerate", "datasets", "bitsandbytes", "triton"])
TRAIN_VERSIONS

In [ ]:
# 2. TEST 1: generate with Transformers on the T4
FALLBACK_1 = "Fix versions/dtype (max 60 min), else switch to text-only Qwen3-4B or Qwen3-1.7B"
model = None
try:
    import torch
    from transformers import AutoTokenizer
    try:
        from transformers import AutoModelForMultimodalLM as ModelClass
    except ImportError:
        from transformers import AutoModelForImageTextToText as ModelClass
    MID = "Qwen/Qwen3.5-2B"
    tok = AutoTokenizer.from_pretrained(MID)
    model = ModelClass.from_pretrained(MID, dtype=torch.float16, device_map="cuda").eval()   # the T4 has no bf16
    EOS = [tok.convert_tokens_to_ids("<|im_end|>"), tok.convert_tokens_to_ids("<|endoftext|>")]

    def reply(user, tools=None, n=150):
        text = tok.apply_chat_template([{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}],
                                       tools=tools, add_generation_prompt=True, tokenize=False)
        ids = tok(text, return_tensors="pt").input_ids.to("cuda")
        t0 = time.time()
        out = model.generate(ids, max_new_tokens=n, do_sample=False, eos_token_id=EOS)[0, ids.shape[1]:]
        return tok.decode(out, skip_special_tokens=False), len(out), time.time() - t0

    chat, n_chat, s_chat = reply("Hi, can I return headphones I bought last week?")
    call, n_call, s_call = reply("Hi, where is my order 4417?", tools=TOOLS)
    ok = chat.rstrip().endswith("<|im_end|>") and len(chat.strip()) > 30
    record("1. Transformers generate on T4", ok,
           f"GPU: {torch.cuda.get_device_name(0)}\n"
           f"chat: {n_chat} tokens in {s_chat:.1f}s ({n_chat / s_chat:.1f} tok/s)\n{chat}\n"
           f"tool prompt: {n_call} tokens in {s_call:.1f}s\n{call}", FALLBACK_1)
except Exception:
    record("1. Transformers generate on T4", False, err(), FALLBACK_1)
finally:
    del model
    gc.collect()
    try:
        torch.cuda.empty_cache()
    except Exception:
        pass

In [ ]:
# 3. TEST 2: 50-step Unsloth LoRA on 200 Bitext rows, save, reload, generate (Unsloth's Qwen3.5 text-only recipe)
FALLBACK_2 = "Switch to text-only Qwen3-4B or Qwen3-1.7B for the project"
TEST_PROMPT = "I want to cancel my order"
try:
    import torch
    from unsloth import FastLanguageModel
    from datasets import load_dataset
    from trl import SFTTrainer, SFTConfig

    model, tok2 = FastLanguageModel.from_pretrained(
        model_name="unsloth/Qwen3.5-2B", max_seq_length=1024,
        load_in_4bit=False, load_in_16bit=True, full_finetuning=False)

    def gen(m, t, user, n=120):
        FastLanguageModel.for_inference(m)
        text = t.apply_chat_template([{"role": "user", "content": user}], add_generation_prompt=True, tokenize=False)
        tk = getattr(t, "tokenizer", t)   # Unsloth may return a multimodal processor; its tokenizer is inside
        ids = t(text=text, return_tensors="pt").input_ids.to("cuda")
        stop = [tk.convert_tokens_to_ids("<|im_end|>"), tk.convert_tokens_to_ids("<|endoftext|>")]
        out = m.generate(input_ids=ids, max_new_tokens=n, do_sample=False, eos_token_id=stop)[0, ids.shape[1]:]
        return tk.decode(out, skip_special_tokens=False)

    before = gen(model, tok2, TEST_PROMPT)

    model = FastLanguageModel.get_peft_model(
        model, r=16, lora_alpha=16, lora_dropout=0, bias="none",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        use_gradient_checkpointing="unsloth", random_state=3407)
    FastLanguageModel.for_training(model)

    ds = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset", split="train[:200]")
    ds = ds.map(lambda r: {"text": tok2.apply_chat_template(
        [{"role": "user", "content": r["instruction"]},
         {"role": "assistant", "content": r["response"]}], tokenize=False)})
    trainer = SFTTrainer(model=model, processing_class=tok2, train_dataset=ds,
        args=SFTConfig(output_dir="spike", per_device_train_batch_size=1,
                       gradient_accumulation_steps=4, max_steps=50,
                       learning_rate=2e-4, logging_steps=5,
                       optim="adamw_8bit", seed=3407, report_to="none"))
    t0 = time.time()
    trainer.train()
    train_s = time.time() - t0
    losses = [round(h["loss"], 3) for h in trainer.state.log_history if "loss" in h]
    model.save_pretrained(ADAPTER_DIR)
    tok2.save_pretrained(ADAPTER_DIR)

    del model, trainer
    gc.collect(); torch.cuda.empty_cache()
    reloaded, tok3 = FastLanguageModel.from_pretrained(
        model_name=ADAPTER_DIR, max_seq_length=1024, load_in_4bit=False, load_in_16bit=True)
    after = gen(reloaded, tok3, TEST_PROMPT)

    loss_falls = len(losses) >= 2 and losses[-1] < losses[0]
    changed = after.strip() != before.strip()
    record("2. Unsloth LoRA: train, save, reload, generate", loss_falls and changed,
           f"50 steps in {train_s:.0f}s | losses every 5 steps: {losses}\n"
           f"loss falls: {loss_falls} | output changed after reload: {changed}\n"
           f"adapter files: {sorted(os.listdir(ADAPTER_DIR))}\n"
           f"--- base model ---\n{before}\n--- reloaded adapter ---\n{after}", FALLBACK_2)
    del reloaded
except Exception:
    record("2. Unsloth LoRA: train, save, reload, generate", False, err(), FALLBACK_2)
finally:
    gc.collect()
    try:
        torch.cuda.empty_cache()
    except Exception:
        pass

In [ ]:
# 4. Install vLLM in its own environment, so it cannot change the training libraries
%pip install -q uv
!uv venv -q /tmp/vllm-env --python 3.12
!uv pip install -q --python /tmp/vllm-env/bin/python "vllm>=0.17"

SERVE_VERSIONS = json.loads(subprocess.run(
    ["/tmp/vllm-env/bin/python", "-c",
     "import importlib.metadata as m, json; print(json.dumps({n: m.version(n) for n in "
     "['vllm', 'torch', 'transformers', 'triton']}))"],
    capture_output=True, text=True).stdout or "{}")
SERVE_VERSIONS

In [ ]:
# 5. Start the vLLM server (flags from the Qwen3.5-2B model card); try with the LoRA adapter, then without
import urllib.request

def start_vllm(with_lora):
    cmd = ["/tmp/vllm-env/bin/vllm", "serve", "Qwen/Qwen3.5-2B", "--dtype", "half", "--max-model-len", "8192",
           "--language-model-only", "--enable-auto-tool-choice", "--tool-call-parser", "qwen3_coder", "--port", "8000"]
    if VLLM_GPU == "0":
        cmd += ["--gpu-memory-utilization", "0.8"]
    if with_lora:
        cmd += ["--enable-lora", "--lora-modules", f"nova-test={ADAPTER_DIR}", "--max-lora-rank", "16"]
    log = open(f"vllm-{'lora' if with_lora else 'base'}.log", "w")
    proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT,
                            env=dict(os.environ, CUDA_VISIBLE_DEVICES=VLLM_GPU))
    t0 = time.time()
    while time.time() - t0 < 1200:                       # first start downloads and compiles: allow 20 min
        if proc.poll() is not None:
            break
        try:
            urllib.request.urlopen("http://localhost:8000/v1/models", timeout=5)
            print(f"vLLM ready in {time.time() - t0:.0f}s (lora={with_lora})")
            return proc, ""
        except Exception:
            time.sleep(10)
    proc.terminate()
    tail = open(log.name).read()[-2500:]
    print(f"vLLM did not start (lora={with_lora}). Log tail:\n{tail}")
    return None, tail

LORA_ERROR = ""
server, LORA_ERROR = start_vllm(with_lora=os.path.isdir(ADAPTER_DIR))
SERVER_HAS_LORA = server is not None and os.path.isdir(ADAPTER_DIR)
if server is None and os.path.isdir(ADAPTER_DIR):        # isolate test 3 from a LoRA-only failure
    server, base_error = start_vllm(with_lora=False)
    if server is None:
        LORA_ERROR += "\n--- without LoRA ---\n" + base_error

In [ ]:
# 6. TEST 3 and TEST 4, plus the model card's sampling settings (repeats the sampling experiment in notes/generation-experiments/)
FALLBACK_3 = "Run evals with Transformers or llama.cpp; serve on an L4- or A10-class GPU"
FALLBACK_4 = "Merge the adapter into the base model instead of serving it separately"
SAMPLING = {}
if server is None:
    record("3. vLLM parsed tool call", False, "vLLM server did not start:\n" + LORA_ERROR, FALLBACK_3)
    record("4. vLLM serves LoRA adapter", False, "vLLM server did not start", FALLBACK_4)
else:
    from openai import OpenAI
    client = OpenAI(base_url="http://localhost:8000/v1", api_key="unused")
    try:
        r = client.chat.completions.create(model="Qwen/Qwen3.5-2B", tools=TOOLS, temperature=0,
            messages=[{"role": "user", "content": "Where is my order 4417?"}])
        calls = r.choices[0].message.tool_calls
        ok = bool(calls) and calls[0].function.name == "get_order_status" \
            and json.loads(calls[0].function.arguments).get("order_id") in (4417, "4417")
        record("3. vLLM parsed tool call", ok, f"tool_calls: {calls}\ncontent: {r.choices[0].message.content!r}", FALLBACK_3)
    except Exception:
        record("3. vLLM parsed tool call", False, err(), FALLBACK_3)

    if not SERVER_HAS_LORA:
        record("4. vLLM serves LoRA adapter", False,
               "Server only started without LoRA.\n" + (LORA_ERROR or "No adapter (test 2 failed)."), FALLBACK_4)
    else:
        try:
            ask = [{"role": "user", "content": "I want to cancel my order"}]
            base = client.chat.completions.create(model="Qwen/Qwen3.5-2B", messages=ask, temperature=0, max_tokens=150)
            lora = client.chat.completions.create(model="nova-test", messages=ask, temperature=0, max_tokens=150)
            b, l = base.choices[0].message.content, lora.choices[0].message.content
            record("4. vLLM serves LoRA adapter", bool(l) and l.strip() != (b or "").strip(),
                   f"--- base ---\n{b}\n--- nova-test ---\n{l}", FALLBACK_4)
        except Exception:
            record("4. vLLM serves LoRA adapter", False, err(), FALLBACK_4)

    try:
        ask = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": "Hi, where is my order 4417?"}]
        for name, kw in {"temperature 0.2": dict(temperature=0.2),
                         "model card (temp 1.0, top_p 1.0, top_k 20, presence_penalty 2.0)":
                             dict(temperature=1.0, top_p=1.0, presence_penalty=2.0, extra_body={"top_k": 20})}.items():
            runs = []
            for seed in range(5):
                c = client.chat.completions.create(model="Qwen/Qwen3.5-2B", messages=ask, max_tokens=600,
                                                   seed=seed, **kw)
                runs.append({"tokens": c.usage.completion_tokens, "finish": c.choices[0].finish_reason,
                             "text": c.choices[0].message.content[:200]})
            SAMPLING[name] = runs
            print(f"\n{name}: lengths {[x['tokens'] for x in runs]}")
            for x in runs:
                print(f"  ({x['tokens']} tokens, {x['finish']}) {x['text']}")
    except Exception:
        print("sampling comparison failed:", err())
    server.terminate()

In [ ]:
# 7. REPORT: copy everything from the next line to the end and paste it back
print("===== RISK TEST REPORT =====")
print(f"GPUs: {N_GPUS} | vLLM on GPU {VLLM_GPU}")
print("\n| Test | Result | Fallback if failed |\n|---|---|---|")
for r in REPORT:
    print(f"| {r['test']} | {r['result']} | {r['fallback']} |")
print("\n--- training environment (tests 1-2) ---")
for k, v in TRAIN_VERSIONS.items():
    print(f"{k}=={v}")
print("\n--- serving environment (tests 3-4) ---")
for k, v in SERVE_VERSIONS.items():
    print(f"{k}=={v}")
print("\n--- details ---")
for r in REPORT:
    print(f"\n## {r['test']}: {r['result']}\n{r['detail']}")
if SAMPLING:
    print("\n--- sampling lengths (vLLM) ---")
    for name, runs in SAMPLING.items():
        print(f"{name}: {[(x['tokens'], x['finish']) for x in runs]}")
print("===== END OF REPORT =====")